# 面试题：有副作用的工具为什么需要幂等键？

可复述答案：网络超时不等于请求没有生效；有副作用请求要携带稳定幂等键，服务端原子记录 key、请求摘要和最终结果。相同 key+相同摘要返回原结果，相同 key+不同摘要明确拒绝。幂等解决重复投递，不解决不同逻辑请求的库存竞争，因此仍需条件更新或 Saga。

## 真实案例

退款 Agent 因网络超时重发请求。六个事件中包含重复投递、相同 key 的金额变化、不同 key 的不同订单。

## 基线

基线每次收到请求都扣减可退款金额。

## 结果解读

手写 ledger 输出命中、冲突或新建三种状态及账本。

## 失败案例

只按 key 去重却不比较请求摘要，会把金额改变的请求错误当成原请求。

In [1]:
events = [{'id':'I01','key':'k1','order':'A12','amount':80}, {'id':'I02','key':'k1','order':'A12','amount':80}, {'id':'I03','key':'k2','order':'A13','amount':30}, {'id':'I04','key':'k1','order':'A12','amount':90}, {'id':'I05','key':'k3','order':'A14','amount':20}, {'id':'I06','key':'k4','order':'A14','amount':20}]  # 构造六个退款投递事件，包括重复与键复用冲突。
print('退款事件:', events)  # 输出包含幂等键、订单和金额的原始事件流。
print('教学说明：key 由调用方为同一逻辑动作稳定生成，不能由每次重试随机生成。')  # 明确幂等键的生成边界。

退款事件: [{'id': 'I01', 'key': 'k1', 'order': 'A12', 'amount': 80}, {'id': 'I02', 'key': 'k1', 'order': 'A12', 'amount': 80}, {'id': 'I03', 'key': 'k2', 'order': 'A13', 'amount': 30}, {'id': 'I04', 'key': 'k1', 'order': 'A12', 'amount': 90}, {'id': 'I05', 'key': 'k3', 'order': 'A14', 'amount': 20}, {'id': 'I06', 'key': 'k4', 'order': 'A14', 'amount': 20}]
教学说明：key 由调用方为同一逻辑动作稳定生成，不能由每次重试随机生成。


In [2]:
unsafe_total = sum(row['amount'] for row in events)  # 计算无去重基线会实际处理的全部金额。
print('无幂等基线总退款:', unsafe_total)  # 输出重复投递造成的过度退款金额。
print('基线问题：I01 与 I02 是同一逻辑退款，却会扣款两次。')  # 点出副作用重复的业务风险。

无幂等基线总退款: 320
基线问题：I01 与 I02 是同一逻辑退款，却会扣款两次。


In [3]:
ledger = {}  # 初始化服务端持久化账本的教学内存版本。
def request_digest(row):  # 定义参与幂等比较的稳定业务摘要。
    return row['order'] + ':' + str(row['amount'])  # 用订单与金额构造最小可解释摘要。
def submit(row):  # 定义服务端的幂等写入状态机。
    digest = request_digest(row)  # 计算当前逻辑请求的摘要。
    if row['key'] not in ledger:  # 处理第一次看到幂等键的请求。
        ledger[row['key']] = {'digest':digest,'result':'退款已创建','amount':row['amount']}  # 原子记录摘要和最终结果。
        return 'created', ledger[row['key']]  # 返回新建结果与保存内容。
    if ledger[row['key']]['digest'] == digest:  # 检查重试请求是否与原请求完全相同。
        return 'replayed', ledger[row['key']]  # 复放原结果而不再次产生副作用。
    return 'conflict', ledger[row['key']]  # 拒绝同键不同摘要的危险复用。

In [4]:
results = [(row['id'],) + submit(row) for row in events]  # 按到达顺序处理六个退款事件。
print('id | 幂等结果 | 账本记录')  # 输出账本状态迁移表标题。
for item in results:  # 遍历每个事件对应的创建、复放或冲突结论。
    print(item[0], item[1], item[2])  # 输出当前事件的服务端状态。
safe_total = sum(item['amount'] for item in ledger.values())  # 汇总去重后真正创建的退款金额。
print('幂等后总退款:', safe_total, '，避免重复金额:', unsafe_total - safe_total)  # 输出相对基线避免的重复副作用。

id | 幂等结果 | 账本记录
I01 created {'digest': 'A12:80', 'result': '退款已创建', 'amount': 80}
I02 replayed {'digest': 'A12:80', 'result': '退款已创建', 'amount': 80}
I03 created {'digest': 'A13:30', 'result': '退款已创建', 'amount': 30}
I04 conflict {'digest': 'A12:80', 'result': '退款已创建', 'amount': 80}
I05 created {'digest': 'A14:20', 'result': '退款已创建', 'amount': 20}
I06 created {'digest': 'A14:20', 'result': '退款已创建', 'amount': 20}
幂等后总退款: 150 ，避免重复金额: 170


In [5]:
key_only = 'replayed' if events[3]['key'] in ledger else 'created'  # 模拟只检查 key 而不检查摘要的错误逻辑。
conflict = results[3][1]  # 读取完整幂等状态机对 I04 的真实结论。
print('失败案例 I04：只看 key=', key_only, '，检查摘要=', conflict)  # 展示金额变更必须拒绝而不是静默复放。
print('生产差距：ledger 必须原子持久化并带租户、动作类型、TTL、请求摘要和最终响应；内存字典仅说明机制。')  # 说明生产一致性要求。

失败案例 I04：只看 key= replayed ，检查摘要= conflict
生产差距：ledger 必须原子持久化并带租户、动作类型、TTL、请求摘要和最终响应；内存字典仅说明机制。


In [6]:
assert results[1][1] == 'replayed'  # 验证完全相同的重试只复放结果。
assert results[3][1] == 'conflict'  # 验证同键不同金额会被明确拒绝。
assert safe_total < unsafe_total  # 验证账本确实避免了重复副作用。